In [ ]:
import sys

In [ ]:
!{sys.executable} -m pip install bio ete4 svgutils

In [ ]:
import os
import seaborn as sns
import re
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from Bio import Phylo
# ETE4 code
from ete4 import PhyloTree
from ete4.treeview import TreeStyle, NodeStyle, AttrFace, TextFace, RectFace
from ete4 import NCBITaxa
import svgutils.transform as sg

In [ ]:
def rgb_to_hex(rgb):
    ''' Convert RGB to Hex'''
    return '#{:02x}{:02x}{:02x}'.format(int(rgb[0] * 255), int(rgb[1] * 255), int(rgb[2] * 255))

def svg_size_to_px(size_str):
    """Convert SVG size to pixels."""
    
    size_str = size_str.strip()
    if size_str.endswith("px"):
        return float(size_str[:-2])
    elif size_str.endswith("mm"):
        return float(size_str[:-2]) * 3.7795275591  # mm → px
    elif size_str.endswith("pt"):
        return float(size_str[:-2]) * 1.3333333     # pt → px

    else:
        return float(''.join(c for c in size_str if c.isdigit() or c == '.'))
        
def layout(node):
    if node.is_leaf:
        try:
            code_part = node.name.split("|")
            code = code_part[1].split("_")[1]
        except:
            return

        d = rank_info.get(code, [None])[0]
        if d is None:
            return
        if d["species"] is None:
            text = f"{d['genus']} (genus); taxID {code}"
        else:
            text = d["species"]
        node.name = text # save tree in NEWICK
        species = TextFace(text, fsize=28)
        species.margin_right = 25
        node.add_face(species, column=0, position="aligned")
        if d["kingdom"] not in taxa_stats["colors"]["kingdom"]:
            taxa_stats["colors"]["kingdom"][d["kingdom"]] = 1
        else:
            taxa_stats["colors"]["kingdom"][d["kingdom"]] += 1

        color = kingdom_colors[d["kingdom"]]
        kingdom_face = RectFace(40, 40, color, color)
        kingdom_face.margin_right = 25
        node.add_face(kingdom_face, column=1, position="aligned")

        phylum_text = TextFace(d["phylum"], fsize=28)
        if phylum_text not in taxa_stats["phylum"]:
            taxa_stats["phylum"][phylum_text] = 1
        else:
            taxa_stats["phylum"][phylum_text] += 1
        phylum_text.margin_right = 25
        node.add_face(phylum_text, column=2, position="aligned")

        order_text = TextFace(d["order"], fsize=28)
        if order_text not in taxa_stats["order"]:
            taxa_stats["order"][order_text] = 1
        else:
            taxa_stats["order"][order_text] += 1
        order_text.margin_right = 25
        node.add_face(order_text, column=3, position="aligned")

        class_text = TextFace(d["class"], fsize=28)
        if class_text not in taxa_stats["class"]:
            taxa_stats["class"][class_text] = 1
        else:
            taxa_stats["class"][class_text] += 1
        class_text.margin_right = 25
        node.add_face(class_text, column=4, position="aligned")


In [ ]:
colors = sns.color_palette("Paired", 12)

In [ ]:
# assess number of taxa and colors
phylum_distribution = {"NA": []}
input_tree_folder = "./trees"
for tree_file in os.listdir(input_tree_folder):
    name = tree_file.split("_")[0]
    file_path = os.path.join(input_tree_folder, tree_file)
    if os.path.isdir(file_path):
        continue
    t = PhyloTree(open(file_path).read(), parser=1)

    # style nodes
    for node in t.traverse():
        if node.is_leaf:
            code_part = node.name.split("|")
            tax_id = code_part[1].split("_")[1]
            if tax_id not in phylum_distribution:
                phylum_distribution[tax_id] = 1
                continue
            phylum_distribution[tax_id] += 1

In [ ]:
ncbi = NCBITaxa()
taxids = list(phylum_distribution.keys())

# Get rank information
rank_info = {}

for taxid in taxids:
    try:
        lineage = ncbi.get_lineage(taxid)
        if not lineage:
            raise ValueError("Empty lineage")

        ranks = ncbi.get_rank(lineage)
        names = ncbi.get_taxid_translator(lineage)

        record = {
            "taxid": taxid,
            "domain": None,
            "kingdom": None,
            "phylum": None,
            "order": None,
            "class": None,
            "family": None,
            "genus": None,
            "species": None     
        }

        for lin_taxid in lineage:
            rank = ranks.get(lin_taxid)
            if rank in ["domain"]:
                record["domain"] = names.get(lin_taxid)
            elif rank == "kingdom":
                record["kingdom"] = names.get(lin_taxid)
            elif rank == "phylum":
                record["phylum"] = names.get(lin_taxid)
            elif rank == "order":
                record["order"] = names.get(lin_taxid)
            elif rank == "class":
                record["class"] = names.get(lin_taxid)
            elif rank == "family":
                record["family"] = names.get(lin_taxid)
            elif rank == "genus":
                record["genus"] = names.get(lin_taxid)
            elif rank == "species":
                record["species"] = names.get(lin_taxid)
        if taxid not in rank_info:
            rank_info[taxid] = [record]
            continue
        rank_info[taxid].append(record)

    except Exception:
        print(f"incorrect id: {taxid}")

In [ ]:
# manual correction for custom taxa
rank_info['999999009'] = [{'taxid': '999999009',
  'domain': 'Eukaryota',
  'kingdom': 'Metazoa',
  'phylum': 'Arthropoda',
  'order': None,
  'class': None,
  'family': 'Autognetidae',
  'genus': 'Autogneta',
  'species': 'Autogneta dalecarlica'}]

rank_info['999999012'] = [{'taxid': '999999012',
  'domain': 'Eukaryota',
  'kingdom': 'Metazoa',
  'phylum': 'Arthropoda',
  'order': None,
  'class': None,
  'family': 'Microzetidae',
  'genus': 'Microzetes',
  'species': 'Microzetes septentrionalis'}]

In [ ]:
phylum = set()
order = set()
class_ = set()
family = set()
species = set()
kingdom = set()
domain = set()

for _, entry in rank_info.items():
    d = entry[0]
    domain.add(d["domain"])
    kingdom.add(d["kingdom"])
    phylum.add(d["phylum"])
    order.add(d["order"])
    class_.add(d["class"])
    family.add(d["family"])
    species.add(d["species"])

In [ ]:
kingdom_colors = {name: rgb_to_hex(colors[i]) for i, name in enumerate(kingdom)}
# set gray color for species which lack information
kingdom_colors[None] = "#D3D3D3"

In [ ]:
trusted_nodes_color = "#98fb98" 
non_trusted_nodes_color = "#ff6347"
taxa_stats = {
            "kingdom": {},
            "colors": {"kingdom": {}},
            "phylum": {},
            "order": {},
            "class": {}
}

for tree_file in os.listdir(input_tree_folder):
    name = tree_file.split("_")[0]
    print(tree_file)
    if not tree_file.endswith(".treefile"):
        continue
    # Load tree
    t = PhyloTree(open(os.path.join(input_tree_folder, tree_file)).read(), parser=1)
    support_threshold = 80
    # Style nodes
    phylum_distribution = {"NA": []}
    taxa_stats = {
                "kingdom": {},
                "colors": {"kingdom": {}},
                "phylum": {},
                "order": {},
                "class": {}
    }
    for node in t.traverse():
        ns = NodeStyle()
        ns["hz_line_width"] = ns["vt_line_width"] = 20
        if not node.is_leaf:
            try:
                node.support = float(node.name.split("/")[1])
                color = trusted_nodes_color if node.support >= support_threshold else non_trusted_nodes_color
                ns["hz_line_color"] = ns["vt_line_color"] = color
            except Exception as err:
                ns["hz_line_color"] = ns["vt_line_color"] = "gray"
            
            node.set_style(ns)
        elif node.is_leaf:
            code_part = node.name.split("|")
            code = code_part[1].split("_")[1]

    ts = TreeStyle()
    ts.mode = "c" # set r for rectanbular shape
    ts.show_leaf_name = False
    ts.layout_fn = layout
    ts.margin_right = 200
    ts.force_topology = False
    ts.complete_branch_lines_when_necessary = False
    ts.draw_guiding_lines = True
    ts.guiding_lines_type = 1
    ts.guiding_lines_color = "gray"
    ts.rotation = 90
    
    orig_tree_width = 1980
    tree_svg_file = f"./output/{name}_rect_tree.svg"
    t.render(tree_svg_file, w=orig_tree_width, units="px", tree_style=ts)
    t.write(outfile=f"./output/{name}_tree_support.nwk", parser=0)

    # Node support legend 
    line_legend_patches = [
        mpatches.Patch(color=trusted_nodes_color, label=f"Bootstrap over {support_threshold}%"),
        mpatches.Patch(color=non_trusted_nodes_color, label=f"Bootstrap below {support_threshold}%")
    ]

    n_rows_line = len(line_legend_patches) + 1
    row_height = 0.4  # inches per row
    fig_height_line = max(2, n_rows_line * row_height)
    fig_width = 4

    fig, ax = plt.subplots(figsize=(fig_width, fig_height_line), dpi=150)
    ax.axis("off")
    ax.legend(handles=line_legend_patches, loc="center", fontsize=12, title="Node Support")
    node_legend_file = f"./output/{name}_node_legend.svg"
    fig.savefig(node_legend_file, format="svg", bbox_inches="tight")
    plt.close(fig)
    sorted_stats = dict(sorted(taxa_stats["colors"]["kingdom"].items(), key=lambda item: item[1], reverse=True))
    legend_patches = [mpatches.Patch(color=kingdom_colors[taxa], label=f"{taxa}: {n} species") 
                                     for taxa, n in sorted_stats.items()
                     ]
    legen_rows = len(legend_patches) + 1
    fig_height = max(2, legen_rows * row_height)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=150)
    ax.axis("off")
    legend = ax.legend(handles=legend_patches, loc="center", fontsize=12, title="Kingdom Colors")
    ax.add_artist(legend)
    phylum_legend_file = f"./output/{name}_phylum_legend.svg"
    fig.savefig(phylum_legend_file, format="svg", bbox_inches="tight")
    plt.close(fig)

    # Combine tree + legends
    fig_tree = sg.fromfile(tree_svg_file)
    fig_node_legend = sg.fromfile(node_legend_file)
    fig_phylum_legend = sg.fromfile(phylum_legend_file)

    w_tree, h_tree = orig_tree_width, orig_tree_width
    w_node, h_node = map(svg_size_to_px, fig_node_legend.get_size())
    w_phylum, h_phylum = map(svg_size_to_px, fig_phylum_legend.get_size())

    padding = 50
    total_width = w_tree + max(w_node, w_phylum) + padding
    total_height = max(h_tree, h_node + h_phylum + padding)

    final_fig = sg.SVGFigure(str(total_width), str(total_height))
    final_fig.root.set("viewBox", f"0 0 {total_width} {total_height}")
    final_fig.root.attrib.pop("width", None)
    final_fig.root.attrib.pop("height", None)
    background = sg.fromstring(
        f'<rect x="0" y="0" width="{total_width}" height="{total_height}" fill="white" />'
    ).getroot()

    final_fig.append([background])

    plot_tree = fig_tree.getroot()
    plot_node = fig_node_legend.getroot()
    plot_phylum = fig_phylum_legend.getroot()

    # Position tree left, legends stacked on right
    plot_tree.moveto(0, 0)
    plot_node.moveto(w_tree + padding, 0)
    plot_phylum.moveto(w_tree + padding, h_node + padding)

    final_fig.append([plot_tree, plot_node, plot_phylum])
    final_fig.save(f"./output/{name}_final.svg")